# 03 — Transformers and Attention from First Principles

**Network LLM Engineering — Part I — Foundations**

### Learning goals
- Understand self-attention and Q/K/V
- Understand causal masking and transformer blocks
- Connect attention to sequence modeling

## Why transformers mattered

Earlier sequence models processed tokens mostly step-by-step. Transformers made **attention** the central operation:
each token representation can compute how strongly it should use information from other token positions.

For a network incident, the word `EXSTART` can attend to an earlier line mentioning `MTU 1500/9216`, while a final diagnosis
can combine evidence spread across the context.

## Self-attention

For hidden vectors `X`, the model projects them into:

- **Q (queries):** what each position is looking for,
- **K (keys):** what each position offers for matching,
- **V (values):** information to aggregate.

Attention is approximately:

`softmax(Q K^T / sqrt(d)) V`

In [ ]:
import torch, math
torch.manual_seed(0)

# 4 "tokens", each with 4-dimensional toy embeddings.
X = torch.randn(4,4)
Wq, Wk, Wv = [torch.randn(4,4) for _ in range(3)]
Q, K, V = X @ Wq, X @ Wk, X @ Wv

scores = Q @ K.T / math.sqrt(Q.shape[-1])
weights = scores.softmax(dim=-1)
output = weights @ V

print("attention weights:")
print(weights.round(decimals=3))
print("\nEach row sums to:", weights.sum(-1))

## Causal masking

Decoder-only LLMs predict the next token. Token position *t* must not see future target tokens.
A **causal mask** sets attention to future positions to effectively negative infinity before softmax.

In [ ]:
scores = torch.zeros(5,5)
mask = torch.triu(torch.ones(5,5), diagonal=1).bool()
scores = scores.masked_fill(mask, float("-inf"))
print(scores)
print("\nsoftmax:")
print(scores.softmax(-1))

## A transformer block

A simplified decoder block is:

`input -> norm -> causal self-attention -> residual -> norm -> MLP/FFN -> residual`

Real models add architectural choices such as RoPE, gated MLPs, grouped-query attention, MoE, and optimized kernels.
You do not need to memorize every variant before fine-tuning, but you must understand the core data flow.

### Exercise

Why would a long `show tech` consume much more memory than a short incident summary?
Hint: consider sequence length, activations, and attention/KV-cache state.